# 🔬 Notebook 08 — Data Quality Audit

**Purpose:** Programmatically detect price outliers and data quality issues across all 3 platform CSVs before trusting any price comparison in the dashboard.

**Method:** IQR (Interquartile Range) outlier detection — standard, non-parametric, robust to non-normal price distributions.

**Output:** `data/clean/data_quality_report.csv` — used by the Data Quality dashboard page.

In [ ]:
# CELL 1 — Imports and load
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df_bl = pd.read_csv('../data/clean/blinkit_clean.csv')
df_ze = pd.read_csv('../data/clean/zepto_clean.csv')
df_bb = pd.read_csv('../data/clean/bigbasket_clean.csv')

print('✅ Data loaded')
print(f'Blinkit: {len(df_bl):,} rows | Zepto: {len(df_ze):,} rows | BigBasket: {len(df_bb):,} rows')
print(f'\nBlinkit columns:   {df_bl.columns.tolist()}')
print(f'Zepto columns:     {df_ze.columns.tolist()}')
print(f'BigBasket columns: {df_bb.columns.tolist()}')

In [ ]:
# CELL 2 — Missing value audit
print('='*60)
print('📋 MISSING VALUE AUDIT')
print('='*60)

platforms = {'Blinkit': df_bl, 'Zepto': df_ze, 'BigBasket': df_bb}
missing_rows = []

for name, df in platforms.items():
    total = len(df)
    print(f'\n{name} ({total:,} rows):')
    for col in df.columns:
        n_missing = df[col].isna().sum()
        pct = n_missing / total * 100
        if n_missing > 0:
            print(f'  {col}: {n_missing:,} missing ({pct:.1f}%)')
            missing_rows.append({'platform': name, 'column': col,
                                  'missing_count': n_missing, 'missing_pct': round(pct, 1)})
    if not any(df[col].isna().sum() > 0 for col in df.columns):
        print(f'  ✅ No missing values')

missing_df = pd.DataFrame(missing_rows)
print(f'\n📊 Total columns with missing data: {len(missing_df)}')

In [ ]:
# CELL 3 — IQR Price Outlier Detection
# Method: Q1 - 3×IQR (lower fence) and Q3 + 3×IQR (upper fence)
# Using 3×IQR (extreme outliers) because retail prices have heavy right tails

print('='*60)
print('📊 IQR PRICE OUTLIER DETECTION (3×IQR extreme threshold)')
print('='*60)

price_col_map = {
    'Blinkit':   ('sale_price', df_bl),
    'Zepto':     ('sale_price', df_ze),
    'BigBasket': ('sale_price', df_bb),
}

outlier_results = []

for platform, (price_col, df) in price_col_map.items():
    if price_col not in df.columns:
        print(f'{platform}: price column "{price_col}" not found — skipping')
        continue

    prices = df[price_col].dropna()
    Q1 = prices.quantile(0.25)
    Q3 = prices.quantile(0.75)
    IQR = Q3 - Q1
    lower_fence = Q1 - 3 * IQR
    upper_fence = Q3 + 3 * IQR

    outliers = df[(df[price_col] < lower_fence) | (df[price_col] > upper_fence)]
    n_outliers = len(outliers)
    pct_outliers = n_outliers / len(df) * 100

    print(f'\n{platform}:')
    print(f'  Q1=₹{Q1:.0f} | Q3=₹{Q3:.0f} | IQR=₹{IQR:.0f}')
    print(f'  Fences: [₹{lower_fence:.0f}, ₹{upper_fence:.0f}]')
    print(f'  Outliers: {n_outliers:,} ({pct_outliers:.1f}%)')
    if n_outliers > 0:
        print(f'  Max outlier: ₹{outliers[price_col].max():,.0f}')
        print(f'  Min outlier: ₹{outliers[price_col].min():,.0f}')
        # Show worst offenders
        worst = outliers.nlargest(5, price_col)[['category', price_col]]
        print(f'  Top 5 extreme outliers:')
        print(worst.to_string(index=False))

    outlier_results.append({
        'platform': platform,
        'total_rows': len(df),
        'price_col': price_col,
        'Q1': round(Q1, 2),
        'Q3': round(Q3, 2),
        'IQR': round(IQR, 2),
        'lower_fence': round(lower_fence, 2),
        'upper_fence': round(upper_fence, 2),
        'n_outliers': n_outliers,
        'pct_outliers': round(pct_outliers, 2),
        'max_price': round(prices.max(), 2),
        'median_price': round(prices.median(), 2),
    })

outlier_df = pd.DataFrame(outlier_results)
print('\n✅ Outlier detection complete')

In [ ]:
# CELL 4 — Category-level median price validation
# Flag categories where median price is suspiciously high (> 5000 INR)
# These are likely data entry errors or currency mismatches in source CSV

SUSPICIOUS_THRESHOLD = 5000  # ₹5,000 median price per category = suspicious for grocery/QC

print('='*60)
print('⚠️  CATEGORY-LEVEL PRICE SANITY CHECK (median > ₹5,000)')
print('='*60)

cat_issues = []

for platform, (price_col, df) in price_col_map.items():
    if price_col not in df.columns or 'category' not in df.columns:
        continue
    cat_medians = df.groupby('category')[price_col].median().reset_index()
    cat_medians.columns = ['category', 'median_price']
    suspicious = cat_medians[cat_medians['median_price'] > SUSPICIOUS_THRESHOLD]

    if len(suspicious) > 0:
        print(f'\n{platform} — {len(suspicious)} suspicious categories:')
        for _, row in suspicious.iterrows():
            print(f'  ⚠️  {row["category"]}: median ₹{row["median_price"]:,.0f}')
            cat_issues.append({
                'platform': platform,
                'category': row['category'],
                'median_price': row['median_price'],
                'issue': f'Median ₹{row["median_price"]:,.0f} exceeds ₹{SUSPICIOUS_THRESHOLD:,} sanity threshold'
            })
    else:
        print(f'{platform}: ✅ All category medians within ₹{SUSPICIOUS_THRESHOLD:,}')

cat_issues_df = pd.DataFrame(cat_issues)
print(f'\n📊 Total category-level issues found: {len(cat_issues_df)}')

In [ ]:
# CELL 5 — Duplicate detection
print('='*60)
print('🔁 DUPLICATE PRODUCT DETECTION')
print('='*60)

dup_results = []
name_col_map = {'Blinkit': 'product_name', 'Zepto': 'name', 'BigBasket': 'product'}

for platform, (price_col, df) in price_col_map.items():
    name_col = name_col_map.get(platform)
    if name_col and name_col in df.columns:
        dups = df.duplicated(subset=[name_col], keep=False).sum()
        pct = dups / len(df) * 100
        print(f'{platform}: {dups:,} duplicate product names ({pct:.1f}%)')
        dup_results.append({'platform': platform, 'duplicate_rows': dups, 'duplicate_pct': round(pct, 1)})
    else:
        # Try generic fallback
        dups = df.duplicated().sum()
        pct = dups / len(df) * 100
        print(f'{platform}: {dups:,} fully duplicate rows ({pct:.1f}%)')
        dup_results.append({'platform': platform, 'duplicate_rows': dups, 'duplicate_pct': round(pct, 1)})

print('\n✅ Duplicate check complete')

In [ ]:
# CELL 6 — Compile full quality report and save
import os
os.makedirs('../data/clean', exist_ok=True)

# Summary report
quality_report = outlier_df.copy()
quality_report.to_csv('../data/clean/data_quality_report.csv', index=False)

# Category issues
if len(cat_issues_df) > 0:
    cat_issues_df.to_csv('../data/clean/price_category_issues.csv', index=False)

print('🎉 DATA QUALITY AUDIT COMPLETE!')
print('='*55)
print('Outputs:')
print('  ✅ data/clean/data_quality_report.csv  — per-platform IQR stats')
print('  ✅ data/clean/price_category_issues.csv — suspicious categories')
print('='*55)
print('\n📊 Quality Report Summary:')
print(quality_report[['platform','total_rows','n_outliers','pct_outliers','median_price']].to_string(index=False))

if len(cat_issues_df) > 0:
    print('\n⚠️  Category-level price issues detected:')
    print(cat_issues_df.to_string(index=False))
    print('\n💡 Recommendation: These categories should be excluded from')
    print('   cross-platform price comparisons until source data is verified.')